In [ ]:
cat << 'EOF' > ~/ros2_ws/src/irb120_description/verificar_hito1.py
import numpy as np
import rclpy
from rclpy.node import Node
from tf2_ros import Buffer, TransformListener

def dh_matrix(theta, d, a, alpha):
    ct, st = np.cos(theta), np.sin(theta)
    ca, sa = np.cos(alpha), np.sin(alpha)
    return np.array([
        [ct, -st * ca,  st * sa, a * ct],
        [st,  ct * ca, -ct * sa, a * st],
        [ 0,       sa,       ca,      d],
        [ 0,        0,        0,      1]
    ])

def forward_kinematics(q):
    # Parametros D-H de tu tabla en metros y radianes
    params = [
        (q[0],             0.290, 0.000, -np.pi/2),
        (q[1] - np.pi/2,   0.000, 0.270,  0.0),
        (q[2],             0.000, 0.070, -np.pi/2),
        (q[3],             0.302, 0.000,  np.pi/2),
        (q[4],             0.000, 0.000, -np.pi/2),
        (q[5],             0.072, 0.000,  0.0)
    ]
    T = np.eye(4)
    for p in params:
        T = T @ dh_matrix(*p)
    return T[0:3, 3]

class VerificadorHito1(Node):
    def __init__(self):
        super().__init__('verificador_hito1')
        self.tf_buffer = Buffer()
        self.tf_listener = TransformListener(self.tf_buffer, self)
        self.timer = self.create_timer(1.0, self.comparar_poses)

    def comparar_poses(self):
        try:
            # Consulta la transformacion publicada por RViz / robot_state_publisher
            t = self.tf_buffer.lookup_transform('base_link', 'tool0', rclpy.time.Time())
            pos_rviz = np.array([
                t.transform.translation.x,
                t.transform.translation.y,
                t.transform.translation.z
            ])

            # Con el simulador en home, los angulos son todos 0
            q_home = np.zeros(6)
            pos_analitica = forward_kinematics(q_home)

            error = np.linalg.norm(pos_rviz - pos_analitica)

            print("\n================ COMPROBACIÓN HITO 1 ================")
            print(f"Pose Analítica (Python D-H) : X={pos_analitica[0]:.6f}, Y={pos_analitica[1]:.6f}, Z={pos_analitica[2]:.6f}")
            print(f"Pose RViz (TF2 URDF)        : X={pos_rviz[0]:.6f}, Y={pos_rviz[1]:.6f}, Z={pos_rviz[2]:.6f}")
            print(f"Error Euclidiano ΔE         : {error:.10e} m")

            if error < 1e-6:
                print("ESTADO: PASAPORTE DE APROBACIÓN CONSEGUIDO (ΔE < 1e-6 m)")
            else:
                print("ESTADO: ERROR DEMASIADO GRANDE, REVISAR OFFSETS")
            print("======================================================")

        except Exception as e:
            print("Esperando transformaciones tf2...")

def main():
    rclpy.init()
    node = VerificadorHito1()
    try:
        rclpy.spin(node)
    except KeyboardInterrupt:
        pass
    node.destroy_node()
    rclpy.shutdown()

if __name__ == '__main__':
    main()
EOF

In [ ]:
'Apertura del proyecto VRiz en ROS'
cd ~/ros2_ws
source /opt/ros/jazzy/setup.bash
source install/setup.bash
ros2 launch irb120_description display.launch.py

In [ ]:
'Llamado para el archivo.py'
cd ~/ros2_ws
source /opt/ros/jazzy/setup.bash
python3 src/irb120_description/verificar_hito1.py

In [ ]:
'''
Ajustar el URDF con precisión analítica XML, en base a los parametros de la Análisis Cinemático.
'''
cat << 'EOF' > ~/ros2_ws/src/irb120_description/urdf/irb120.urdf
<?xml version="1.0"?>
<robot name="abb_irb120">

  <!-- Link Base -->
  <link name="base_link">
    <visual>
      <origin xyz="0 0 0.145" rpy="0 0 0"/>
      <geometry>
        <cylinder radius="0.08" length="0.290"/>
      </geometry>
      <material name="abb_orange"><color rgba="1.0 0.4 0.0 1.0"/></material>
    </visual>
  </link>

  <!-- Joint 1: rotación Z -->
  <joint name="joint_1" type="revolute">
    <parent link="base_link"/>
    <child link="link_1"/>
    <origin xyz="0 0 0.290" rpy="0 0 0"/>
    <axis xyz="0 0 1"/>
    <limit lower="-2.87979" upper="2.87979" effort="100.0" velocity="4.3633"/>
  </joint>

  <link name="link_1"/>

  <!-- Joint 2: eje Y en robot industrial (equivale a D-H tras rotación pi/2) -->
  <joint name="joint_2" type="revolute">
    <parent link="link_1"/>
    <child link="link_2"/>
    <origin xyz="0 0 0" rpy="-1.5707963267948966 0 0"/>
    <axis xyz="0 0 1"/>
    <limit lower="-1.91986" upper="1.91986" effort="100.0" velocity="4.3633"/>
  </joint>

  <link name="link_2">
    <visual>
      <origin xyz="0.135 0 0" rpy="0 1.5707963267948966 0"/>
      <geometry>
        <cylinder radius="0.04" length="0.270"/>
      </geometry>
      <material name="abb_white"><color rgba="0.9 0.9 0.9 1.0"/></material>
    </visual>
  </link>

  <!-- Joint 3 -->
  <joint name="joint_3" type="revolute">
    <parent link="link_2"/>
    <child link="link_3"/>
    <origin xyz="0 0.270 0" rpy="0 0 0"/>
    <axis xyz="0 0 1"/>
    <limit lower="-1.91986" upper="1.22173" effort="100.0" velocity="4.3633"/>
  </joint>

  <link name="link_3"/>

  <!-- Joint 4 -->
  <joint name="joint_4" type="revolute">
    <parent link="link_3"/>
    <child link="link_4"/>
    <origin xyz="0.070 0 0" rpy="-1.5707963267948966 0 0"/>
    <axis xyz="0 0 1"/>
    <limit lower="-2.79253" upper="2.79253" effort="100.0" velocity="5.5850"/>
  </joint>

  <link name="link_4">
    <visual>
      <origin xyz="0 0 0.151" rpy="0 0 0"/>
      <geometry>
        <cylinder radius="0.03" length="0.302"/>
      </geometry>
      <material name="abb_orange"/>
    </visual>
  </link>

  <!-- Joint 5 -->
  <joint name="joint_5" type="revolute">
    <parent link="link_4"/>
    <child link="link_5"/>
    <origin xyz="0 0 0.302" rpy="1.5707963267948966 0 0"/>
    <axis xyz="0 0 1"/>
    <limit lower="-2.09439" upper="2.09439" effort="100.0" velocity="5.5850"/>
  </joint>

  <link name="link_5"/>

  <!-- Joint 6 -->
  <joint name="joint_6" type="revolute">
    <parent link="link_5"/>
    <child link="link_6"/>
    <origin xyz="0 0 0" rpy="-1.5707963267948966 0 0"/>
    <axis xyz="0 0 1"/>
    <limit lower="-6.98132" upper="6.98132" effort="100.0" velocity="7.3304"/>
  </joint>

  <link name="link_6"/>

  <!-- Joint Tool0 (Brida del TCP) -->
  <joint name="joint_tool0" type="fixed">
    <parent link="link_6"/>
    <child link="tool0"/>
    <origin xyz="0 0 0.072" rpy="0 0 0"/>
  </joint>

  <link name="tool0">
    <visual>
      <origin xyz="0 0 -0.01" rpy="0 0 0"/>
      <geometry>
        <cylinder radius="0.02" length="0.02"/>
      </geometry>
      <material name="abb_white"/>
    </visual>
  </link>

</robot>
EOF